Visualize a humanoid rollout from `main_humanoid_corrected_mo_continue.py`.

The wrapper returns a `GeneralizedState`; the Brax env state is `state.env_state`, and the pose used for rendering is `state.env_state.pipeline_state`.

In [ ]:
import jax
import jax.numpy as jnp
import flax.linen as nn
import numpy as np
from pathlib import Path
from brax import envs
from brax.io import html
from flax import serialization
from IPython.display import IFrame

from networks import GC_PPO_Policy
from task_wrappers.humanoid_corrected_mo_wrapper import HumanoidMOWrapper
# from task_wrappers.humanoid_mo_wrapper import HumanoidMOWrapper
from typing import Tuple

In [ ]:
%matplotlib inline

In [ ]:
action_repeat = 3
episode_length = 4096
# folder_path = "./output/MORL/test6"
folder_path = "output/MORL/huamnoid_mo_corrected"

env = envs.create(
    env_name="humanoid",
    episode_length=episode_length,
    backend="mjx",
    auto_reset=False,
    action_repeat=action_repeat,
)
env = HumanoidMOWrapper(env)

jit_reset = jax.jit(env.reset)
jit_step = jax.jit(env.step)

In [ ]:
actor_hidden_layers: Tuple[int, ...] = (256, 256)
policy_network = GC_PPO_Policy(
    hidden_layer_sizes=actor_hidden_layers,
    action_dim=env.action_size,
    initial_std=0.1 * jnp.ones(env.action_size),
    kernel_init=jax.nn.initializers.orthogonal(jnp.sqrt(2)),
    kernel_init_final=jax.nn.initializers.orthogonal(0.01),
    activation=nn.silu,
    final_activation=jnp.tanh,
    learnable_std=True,
)

seed = 8848
key = jax.random.PRNGKey(seed)
key, subkey = jax.random.split(key)

fake_obs = jnp.zeros(env.observation_size)
fake_z = jnp.zeros(env.z_size)
policy_template = policy_network.init(subkey, obs=fake_obs, z=fake_z)

with open(folder_path + "/policy.msgpack", "rb") as f:
    policy_params = serialization.from_bytes(policy_template, f.read())

jit_policy = jax.jit(lambda obs, z: policy_network.apply(policy_params, obs, z))

In [ ]:
key, subkey = jax.random.split(key)
state = jit_reset(subkey)

# Optional: override the sampled preference (vx, vy, height, control, consistency).
# Comment this block out to keep the random preference from reset.
preference = jnp.array([1.0, 0.0, 0.3, 0.2, 0.2])
preference = preference / jnp.sqrt(jnp.sum(jnp.square(preference)) + 1e-6)
z_state = state.z_state.replace(
    preference=preference,
    z=jnp.concatenate([state.z_state.last_action, preference]),
)
state = state.replace(z_state=z_state)
print("preference", np.array(state.z_state.preference))

In [ ]:
episode_steps = 300
deterministic = True

trajectory = [state.env_state.pipeline_state]
print(f"Simulating {episode_steps} steps...")

for _ in range(episode_steps):
    obs, z = env.get_obs(state)
    action_mean, std_logits = jit_policy(obs, z)
    if deterministic:
        action = jnp.clip(action_mean, -1.0, 1.0)
    else:
        key, subkey = jax.random.split(key)
        action_std = nn.sigmoid(std_logits)
        action = jnp.clip(
            action_mean + action_std * jax.random.normal(subkey, action_mean.shape),
            -1.0,
            1.0,
        )
    state, _ = jit_step(state, action)
    trajectory.append(state.env_state.pipeline_state)
    if float(state.env_state.done) > 0.5:
        print("episode ended at step", len(trajectory) - 1)
        break

print("collected", len(trajectory), "frames")

In [ ]:
# html.render() returns a full HTML document with <script type="module"> and CDN
# imports. Colab sandboxes cell output in an iframe, so that document can run.
# Jupyter / Cursor injects the HTML into the notebook page, which strips or
# refuses those scripts — so HTML(...) looks like it ran but shows nothing.
html_path = Path("humanoid_rollout.html")
html_path.write_text(html.render(env.sys, trajectory))
IFrame(src=html_path.name, width="100%", height=500)